In [47]:
import os
import json
import numpy as np
import pandas as pd

folder_path = r"D:\OneDrive\Trading\Market Making\data\runs\run_20260601_122340"

snapshots = pd.read_parquet(os.path.join(folder_path, "snapshots.parquet"))
trades = pd.read_parquet(os.path.join(folder_path, "trades.parquet"))
quotes = pd.read_parquet(os.path.join(folder_path, "quotes.parquet"))
fills = pd.read_parquet(os.path.join(folder_path, "fills.parquet"))

events = pd.read_parquet(os.path.join(folder_path, "events.parquet"))
with open(os.path.join(folder_path, "orderbook_snapshot.json"), "r") as f:
    orderbook_snapshot = json.load(f)

In [ ]:
def label_adverse_selection_multi(fills: pd.DataFrame, snapshots: pd.DataFrame, horizons_ms=(100, 500, 1000)):
    fills = fills.sort_values("ts").copy()
    snapshots = snapshots.sort_values("ts")

    for h in horizons_ms:
        future = snapshots[["ts", "mid"]].copy()
        future["ts"] = future["ts"] - h
        future = future.rename(columns={"mid": f"future_mid_{h}"})

        merged = pd.merge_asof(
            fills,
            future.sort_values("ts"),
            on="ts",
            direction="forward"
        )

        raw_move = merged[f"future_mid_{h}"] - fills["price"]

        fills[f"adverse_{h}ms"] = np.where(
            fills["side"] == "BUY",
            -raw_move,
            raw_move
        )

    return fills

def add_trade_impact_multi(trades: pd.DataFrame, snapshots: pd.DataFrame, horizons_ms=(100, 500, 1000)):
    """
    Adds future return at multiple time horizons using snapshots.
    Assumes both inputs are already sorted by ts.
    """

    out = trades.copy()

    for h in horizons_ms:

        future = snapshots[["ts", "mid"]].copy()
        future["ts"] = future["ts"] - h
        future = future.rename(columns={"mid": f"future_mid_{h}"})

        merged = pd.merge_asof(
            out,
            future,
            on="ts",
            direction="forward"
        )

        out[f"trade_return_{h}ms"] = (
            (merged[f"future_mid_{h}"] - out["price"]) / out["price"]
        )

    return out

def label_toxicity_multi(trades: pd.DataFrame, snapshots: pd.DataFrame, horizons_ms=(100, 500, 1000)):
    """
    Toxicity = price moved against aggressor after trade.
    Multi-horizon version.
    Assumes sorted inputs.
    """

    out = trades.copy()

    for h in horizons_ms:

        future = snapshots[["ts", "mid"]].copy()
        future["ts"] = future["ts"] - h
        future = future.rename(columns={"mid": f"future_mid_{h}"})

        merged = pd.merge_asof(
            out,
            future,
            on="ts",
            direction="forward"
        )

        move = merged[f"future_mid_{h}"] - out["price"]

        out[f"toxicity_{h}ms"] = np.where(
            out["side"] == "BUY",
            move < 0,
            move > 0
        ).astype(int)

        out[f"signed_impact_{h}ms"] = np.where(
            out["side"] == "BUY",
            -move,
            move
        )

        out[f"signed_impact_bps_{h}ms"] = (
            10000 * out[f"signed_impact_{h}ms"] / out["price"]
        )

    return out

def align_trades_to_snapshots(trades: pd.DataFrame, snapshots: pd.DataFrame):

    snap = snapshots[["ts", "mid", "best_bid", "best_ask"]].rename(columns={
        "mid": "snap_mid",
        "best_bid": "snap_bid",
        "best_ask": "snap_ask"
    })

    merged = pd.merge_asof(trades, snap, on="ts", direction="backward")

    trades = merged.copy()

    trades["snapshot_mid"] = trades["snap_mid"]
    trades["snapshot_bid"] = trades["snap_bid"]
    trades["snapshot_ask"] = trades["snap_ask"]
    trades["spread"] = trades["snap_ask"] - trades["snap_bid"]

    return trades

def label_fill_markouts_ms(fills, snapshots, horizons_ms=(100, 500, 1000, 5000)):
    # ASSUMES sorted inputs

    snap_ts = snapshots["ts"].values
    snap_mid = snapshots["mid"].values

    fills = fills.copy()

    base_idx = np.searchsorted(snap_ts, fills["ts"].values, side="right") - 1
    base_idx = np.clip(base_idx, 0, len(snap_ts) - 1)

    fills["snap_mid"] = snap_mid[base_idx]

    fill_ts = fills["ts"].values
    fill_price = fills["price"].values

    for h in horizons_ms:
        target = fill_ts + h

        idx = np.searchsorted(snap_ts, target, side="left")
        idx = np.clip(idx, 0, len(snap_ts) - 1)

        fills[f"markout_{h}ms"] = snap_mid[idx] - fill_price

    return fills

def finalize_returns(snapshots, horizons_ms=(100, 500, 1000, 5000)):

    ts = snapshots["ts"].values
    mid = snapshots["mid"].values

    for h in horizons_ms:
        target = ts + h

        idx = np.searchsorted(ts, target, side="left")
        idx = np.clip(idx, 0, len(ts) - 1)

        future_mid = mid[idx]

        snapshots[f"future_mid_{h}ms"] = future_mid
        snapshots[f"future_return_{h}ms"] = (future_mid - mid) / mid

    return snapshots

def generate_datasets(snapshots, trades, quotes, fills):
    snapshots = snapshots.sort_values("ts").copy()
    trades = trades.sort_values("ts").copy()
    quotes = quotes.sort_values("ts").copy()
    fills = fills.sort_values("ts").copy()

    fills = label_adverse_selection_multi(fills, snapshots, horizons_ms=(100, 500, 1000))
    fills = label_fill_markouts_ms(fills, snapshots, horizons_ms=(100, 500, 1000, 5000))
    
    trades = add_trade_impact_multi(trades, snapshots, horizons_ms=(100, 500, 1000))
    trades = label_toxicity_multi(trades, snapshots, horizons_ms=(100, 500, 1000))
    trades = align_trades_to_snapshots(trades, snapshots)

    snapshots = finalize_returns(snapshots, horizons_ms=(100, 500, 1000, 5000))

    return snapshots, trades, quotes, fills

snapshots, trades, quotes, fills = generate_datasets(snapshots, trades, quotes, fills)